# IT Support Dashboard - Supporting Notebook 1

This notebook is the project entry point for data quality assurance. Within the wider IT support ticket analysis workflow, it preserves a traceable record ID, removes unusable text records, corrects malformed tags, standardises key fields, and produces the clean dataset used by the downstream representativeness, lemmatisation, clustering, and reporting notebooks.

## Cleaning Process

Notebook 1 documents the project-wide cleaning and validation layer for the raw ticket export. The following actions were taken:
- Preserved the source row index as an explicit ID field for auditability across exports
- Identified and removed 7 records with null values in `Answer`
- Identified and removed 3,838 records with null values in `Subject`
- Corrected 13 data-entry errors within the tag columns
- Standardised entries within the Priority and Language columns
- Removed formatting artefacts from long-text fields
- Amended data types across analytical dimensions
- Produced clean CSV and Parquet outputs for downstream notebooks
- Saved interim audit tables showing which records were removed or amended
- Exported CSV files without an extra unnamed pandas index column

### Importing

First, the required packages for this process were imported, the necessary folders were created using the config file, and the raw dataset was read.

In [1]:
# Importing essential packages
import sys
from pathlib import Path
import pandas as pd
import os

# Add project root to sys.path by searching upward for config.py
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists()), None
)
if project_root is None:
    raise FileNotFoundError("Could not locate project root containing config.py")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config import Config  # noqa: E402

Config.ensure_directories()

# Reading the initial dataset as a DataFrame: df
# Preserve the original row position as an explicit record identifier for audit outputs.
df = pd.read_csv(
    Config.RAW_DATA_PATH,
    encoding="utf-8",
    na_values=[
        "",
        " ",
        "NA",
        "N/A",
        "na",
        "n/a",
        "NULL",
        "null",
        "None",
        "none",
        "NAN",
        "NaN",
        "nan",
        None,
    ],
).reset_index(names="index")

# Display the fetched table
print("\nThe imported raw dataset:")
display(df)

[Config] Verified project directory structure under C:\Users\David\Desktop\Python_Files\IT-Support-Ticket-Analysis

The imported raw dataset:


,index,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28582,28582,Performance Problem with Data Analytics Tool,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,high,en,400,Performance,IT,Tech Support,NaN,NaN,NaN,NaN,NaN
28583,28583,Datensperrung in der Kundschaftsbetreuung,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,high,de,400,Security,IT,Tech Support,Bug,NaN,NaN,NaN,NaN
28584,28584,Problem mit der Videokonferenz-Software heute,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,low,de,400,Bug,Performance,Network,IT,Tech Support,NaN,NaN,NaN
28585,28585,Update Request for SaaS Platform Integration F...,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,high,en,400,Feature,IT,Tech Support,NaN,NaN,NaN,NaN,NaN


### Assessing the cleanliness of the dataset

Second, we assessed how clean and complete this dataset was by reviewing the table shape, missing values, data types, and memory usage.

In [2]:
# Information about the dataset
print("\nWhat is the shape of my table?")
print(df.shape)
print("\nAre there any missing values in each dimension?")
print(df.isna().sum().sort_values())
print("\nWhat is the datatype of each column?")
print(df.dtypes)
print("\nHow many bytes does each column use?")
print(df.memory_usage(deep=True))


What is the shape of my table?
(28587, 17)

Are there any missing values in each dimension?
index           0
tag_1           0
language        0
priority        0
queue           0
version         0
body            0
type            0
answer          7
tag_2          13
tag_3         136
tag_4        3058
subject      3838
tag_5       14042
tag_6       22713
tag_7       26547
tag_8       28022
dtype: int64

What is the datatype of each column?
index        int64
subject     object
body        object
answer      object
type        object
queue       object
priority    object
language    object
version      int64
tag_1       object
tag_2       object
tag_3       object
tag_4       object
tag_5       object
tag_6       object
tag_7       object
tag_8       object
dtype: object

How many bytes does each column use?
Index            132
index         228696
subject      2489723
body        12609156
answer      12608655
type         1609416
queue        1877997
priority     1532247
languag

### Removing null responses

The following columns contained null values that would impede the downstream text analysis workflow:
- 7 records in `Answer`
- 3,838 records in `Subject`
- 1 record (source ID 13,651) had null values in both columns

Rows containing null values in either column were removed because they would weaken later text modelling steps. The explicit `index` field was retained so that removed records could still be traced back to the source extract. In total, 3,844 records (13.45% of the dataset) were removed at this stage.

In [3]:
# Identify rows with null answers or subjects
null_mask = df["answer"].isna() | df["subject"].isna()
null_answers = df[null_mask].copy().sort_values("index")

# Save the null answers table to CSV without an extra pandas index column
null_answers.to_csv(Config.NULL_ANSWERS_PATH, index=False, encoding="latin-1")

# Count the number of null answers
null_count = null_answers.shape[0]

# Percentage of null answers
percentage_null = 100 * null_count / df.shape[0]

# Show the affected entries
print(
    f"\nTable containing {null_count:,} records with null values, accounting for {percentage_null:.2f}% of the total data:"
)
display(null_answers[["index", "subject", "answer"]])


Table containing 3,844 records with null values, accounting for 13.45% of the total data:


,index,subject,answer
870,870,NaN,Please provide detailed integration instructio...
887,887,NaN,"<name>, I regret to hear about the security br..."
888,888,NaN,Überprüfen Sie die Kampagnenberichte und verei...
915,915,NaN,"<name>, thank you for your email regarding the..."
929,929,NaN,We have received a report about a security inc...
...,...,...,...
28550,28550,NaN,We appreciate you bringing this critical issue...
28552,28552,NaN,"geehrter [name], wir danken Ihnen für Ihre Anf..."
28568,28568,NaN,"Wir bieten Social-Media-Management, Suchmaschi..."
28569,28569,NaN,Wir haben Ihre E-Mail über den Leistungsriss b...


In [4]:
# Remove rows with null values in Answer or Subject while preserving original row indices
dropped_nulls_df = df.copy().dropna(subset=["answer", "subject"])

rows_removed = df.shape[0] - dropped_nulls_df.shape[0]

# Validate amendments
print(
    f"\nTotal rows before null removal: {df.shape[0]:,}; "
    f"after null removal: {dropped_nulls_df.shape[0]:,}; "
    f"rows removed: {rows_removed:,}"
)
print(
    f"\nNull values in Answer before: {df['answer'].isna().sum():,}; "
    f"after removal: {dropped_nulls_df['answer'].isna().sum():,}"
)
print(
    f"\nNull values in Subject before: {df['subject'].isna().sum():,}; "
    f"after removal: {dropped_nulls_df['subject'].isna().sum():,}"
)


Total rows before null removal: 28,587; after null removal: 24,743; rows removed: 3,844

Null values in Answer before: 7; after removal: 0

Null values in Subject before: 3,838; after removal: 0


### Amending tags

Next, anomalies within the tag fields were corrected. In the `tag_1` column, 12 records contained multiple comma-separated tags. Because corresponding entries in subsequent tag columns were empty, these were treated as data-entry errors. The tags were split and distributed across the available tag columns for each affected record.

In the `tag_3` column, 1 record showed a similar validation issue. Those tags were split and redistributed into subsequent fields while preserving the original order.

No issues were identified in the remaining tag columns.

In [5]:
# Count and sort tag columns by numeric suffix
tag_columns = [col for col in dropped_nulls_df.columns if col.startswith("tag_")]
tag_columns = sorted(
    tag_columns,
    key=lambda col: int(col.split("_")[1]) if col.split("_")[1].isdigit() else col,
)
print(f"\nNumber of tag columns: {len(tag_columns)}")

long_tags_dict = {}

# Loop to check for multiple tags
for tag_col in tag_columns:
    has_multiple_tags = dropped_nulls_df[tag_col].str.contains(",", na=False)
    long_tags_df = dropped_nulls_df[has_multiple_tags].sort_values("index")

    if long_tags_df.empty:
        print(f"\nNo rows with multiple tags found in {tag_col}.")
    else:
        # Create a series of indexes of all affected records
        index_list = dropped_nulls_df[has_multiple_tags].index.tolist()
        long_tags_dict[tag_col] = index_list

        print(
            f"\nTable containing {len(index_list)} row(s) with multiple tags within {tag_col}:"
        )
        display(long_tags_df[["index", *tag_columns]])

# Summary of columns with multiple tags
print(
    f"\nColumns {list(long_tags_dict.keys())} contain multiple tags, separated by commas."
)

# Saving the table of rows with multiple tags to a CSV file
# flatten the dict-of-lists into a single list of row numbers
all_indices = sorted(set(idx for lst in long_tags_dict.values() for idx in lst))
dropped_nulls_df.loc[all_indices].sort_values("index").to_csv(
    Config.INVALID_TAGS_PATH, index=False, encoding="latin-1"
)


Number of tag columns: 8

Table containing 12 row(s) with multiple tags within tag_1:


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
958,958,"Performance,Bug,Disruption,Security",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1784,1784,"Performance,Disruption,Outage,Monitoring,Analysis",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2166,2166,"Crash,Performance,Outage,Disruption,Recovery,S...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2331,2331,"Performance,Disruption,IT,Tech Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2410,2410,"Performance,Disruption,Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3146,3146,"Performance,Outage,Disruption,Recovery,Marketi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3537,3537,"Security,IT,Tech Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4758,4758,"Performance,Disruption,Outage,Support,Integration",NaN,NaN,NaN,NaN,NaN,NaN,NaN
5355,5355,"Performance,Security,Feature,Documentation",NaN,NaN,NaN,NaN,NaN,NaN,NaN
5698,5698,"Security,IT,Tech Support,Data Privacy,Regulati...",NaN,NaN,NaN,NaN,NaN,NaN,NaN



No rows with multiple tags found in tag_2.

Table containing 1 row(s) with multiple tags within tag_3:


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
5885,5885,Security,Network,"Disruption,IT",Tech Support,NaN,NaN,NaN,NaN



No rows with multiple tags found in tag_4.

No rows with multiple tags found in tag_5.

No rows with multiple tags found in tag_6.

No rows with multiple tags found in tag_7.

No rows with multiple tags found in tag_8.

Columns ['tag_1', 'tag_3'] contain multiple tags, separated by commas.


In [6]:
# Fixing tags dataframe
fixed_tags_df = dropped_nulls_df.copy()

# Rebuild tag columns row-wise for affected records to preserve order and avoid overwrites
max_tag_cols = len(tag_columns)
overflow_rows = []

for idx in all_indices:
    row_tags = []

    # Collect all tags from tag columns, splitting comma-separated values where needed
    for col in tag_columns:
        value = fixed_tags_df.at[idx, col]
        if pd.isna(value):
            continue
        parts = [part.strip() for part in str(value).split(",") if part.strip()]
        row_tags.extend(parts)

    # Track any tags that cannot fit in available tag columns
    if len(row_tags) > max_tag_cols:
        overflow_rows.append((idx, row_tags[max_tag_cols:]))

    # Fit tags into available columns and pad remaining slots with nulls
    row_tags = row_tags[:max_tag_cols]
    row_tags += [None] * (max_tag_cols - len(row_tags))
    fixed_tags_df.loc[idx, tag_columns] = row_tags

# Checking the affected records to verify whether the changes were successfully implemented
print("\nUpdated rows with multiple tags:")
display(fixed_tags_df.loc[all_indices].sort_values("index")[["index", *tag_columns]])

if overflow_rows:
    print(
        f"\nWarning: {len(overflow_rows)} row(s) had more tags than available columns and were truncated."
    )

# Saving the amended tags table to CSV
fixed_tags_df.loc[all_indices].sort_values("index").to_csv(
    Config.VALIDATED_TAGS_PATH, index=False, encoding="latin-1"
)


Updated rows with multiple tags:


,index,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
958,958,Performance,Bug,Disruption,Security,None,None,None,None
1784,1784,Performance,Disruption,Outage,Monitoring,Analysis,None,None,None
2166,2166,Crash,Performance,Outage,Disruption,Recovery,Server,DataProcessing,None
2331,2331,Performance,Disruption,IT,Tech Support,None,None,None,None
2410,2410,Performance,Disruption,Support,None,None,None,None,None
3146,3146,Performance,Outage,Disruption,Recovery,Marketing,Agentur,Analyse,None
3537,3537,Security,IT,Tech Support,None,None,None,None,None
4758,4758,Performance,Disruption,Outage,Support,Integration,None,None,None
5355,5355,Performance,Security,Feature,Documentation,None,None,None,None
5698,5698,Security,IT,Tech Support,Data Privacy,Regulation,Patient Data,Threat Prevention,None


### Standardising inputs

The dataset was then standardised by normalising variations in the Priority and Language columns, removing formatting artefacts from the Answer and Body fields, and converting data types to reduce memory usage and improve downstream performance. The explicit `index` field was retained as a stable record identifier, while the temporary pandas row index remained excluded from exports.

In [7]:
# Standardising text dataframe
standardised_text_df = fixed_tags_df.copy()

# Standardise text in Priority to title case and Language to upper case
standardised_text_df["priority"] = standardised_text_df["priority"].str.title()
standardised_text_df["language"] = standardised_text_df["language"].str.upper()

# Removing text formatting from body and answer columns
columns_to_clean = ["answer", "body"]
for column in columns_to_clean:
    standardised_text_df[column] = (
        standardised_text_df[column]
        .str.replace(r"(?i)<br\s*/?>", " ", regex=True)
        .str.replace(r"(?:\\r\\n|\\n|\\r|\r\n|\n|\r)", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Capitalise the first letter of the headers of all columns
standardised_text_df.rename(columns=lambda col: col.capitalize(), inplace=True)

# Converting data types to save on memory usage
# Lists of data types by column
categories = [
    "Type",
    "Queue",
    "Priority",
    "Language",
    "Version",
    "Tag_1",
    "Tag_2",
    "Tag_3",
    "Tag_4",
    "Tag_5",
    "Tag_6",
    "Tag_7",
    "Tag_8",
]
strings = ["Subject", "Body", "Answer"]

# Apply dtype conversions in one pass for matching columns only
dtype_map = {col: "category" for col in categories}
dtype_map.update({col: "string" for col in strings})
applicable_dtypes = {
    col: dtype
    for col, dtype in dtype_map.items()
    if col in standardised_text_df.columns
}
standardised_text_df = standardised_text_df.astype(applicable_dtypes)

# Displaying the DataFrame after dropping null answers and converting data types
display(standardised_text_df)

,Index,Subject,Body,Answer,Type,Queue,Priority,Language,Version,Tag_1,Tag_2,Tag_3,Tag_4,Tag_5,Tag_6,Tag_7,Tag_8
0,0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team, ich möchte einen g...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,High,DE,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,1,Account Disruption,"Dear Customer Support Team, I am writing to re...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,High,EN,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team, I hope this messag...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,Medium,EN,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,3,Inquiry Regarding Invoice Details,"Dear Customer Support Team, I hope this messag...",We appreciate you reaching out with your billi...,Request,Billing and Payments,Low,EN,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,4,Question About Marketing Agency Software Compa...,"Dear Support Team, I hope this message reaches...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,Medium,EN,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28582,28582,Performance Problem with Data Analytics Tool,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,High,EN,400,Performance,IT,Tech Support,NaN,NaN,NaN,NaN,NaN
28583,28583,Datensperrung in der Kundschaftsbetreuung,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,High,DE,400,Security,IT,Tech Support,Bug,NaN,NaN,NaN,NaN
28584,28584,Problem mit der Videokonferenz-Software heute,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,Low,DE,400,Bug,Performance,Network,IT,Tech Support,NaN,NaN,NaN
28585,28585,Update Request for SaaS Platform Integration F...,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,High,EN,400,Feature,IT,Tech Support,NaN,NaN,NaN,NaN,NaN


### Validation

The dataset was revalidated to confirm that all changes had been applied successfully.

In [8]:
print("Dataset shape:")
print(standardised_text_df.shape)
print("\nMissing values by column:")
print(standardised_text_df.isna().sum().sort_values())
print("\nData type by column:")
print(standardised_text_df.dtypes)
print("\nMemory usage by column (bytes):")
print(standardised_text_df.memory_usage(deep=True))

Dataset shape:
(24743, 17)

Missing values by column:
Index           0
Tag_2           0
Tag_1           0
Language        0
Priority        0
Version         0
Type            0
Answer          0
Body            0
Subject         0
Queue           0
Tag_3          96
Tag_4        2629
Tag_5       12322
Tag_6       19879
Tag_7       23147
Tag_8       24333
dtype: int64

Data type by column:
Index                int64
Subject     string[python]
Body        string[python]
Answer      string[python]
Type              category
Queue             category
Priority          category
Language          category
Version           category
Tag_1             category
Tag_2             category
Tag_3             category
Tag_4             category
Tag_5             category
Tag_6             category
Tag_7             category
Tag_8             category
dtype: object

Memory usage by column (bytes):
Index         726368
Index         197944
Subject      2366005
Body        10681870
Answer      108

To complete validation, the number of unique values in each column was reviewed.

In [9]:
# Provide a compact unique-value summary for each column
for column in standardised_text_df.columns:
    distinct_count = standardised_text_df[column].nunique(dropna=False)
    top_value_counts = standardised_text_df[column].value_counts(dropna=False).head(10)

    print(f"\n{column}: {distinct_count:,} unique values")
    print("Top 10 values (including nulls):")
    print(top_value_counts)


Index: 24,743 unique values
Top 10 values (including nulls):
Index
0        1
19027    1
19000    1
18998    1
18997    1
18996    1
18995    1
18994    1
18993    1
18992    1
Name: count, dtype: int64

Subject: 24,743 unique values
Top 10 values (including nulls):
Subject
Wesentlicher Sicherheitsvorfall                            1
Possible Data Leak in Hospital IT System                   1
Report on Security Breach Received                         1
Healthcare Provider System Security Incident Identified    1
Unusual Decline in Brand Engagement Metrics Lately         1
Assistance with Support                                    1
Probleme bei der Anmeldung - Benutzerkonten                1
Medizinische Datensperre                                   1
Support for Data Breach Involving Medical Records          1
Assistance with Analytics Investment Tool Performance      1
Name: count, dtype: Int64

Body: 24,741 unique values
Top 10 values (including nulls):
Body
Assistance needed     

### Saving the clean dataset

Two versions were created for downstream use: CSV and Parquet. The CSV file supports broad compatibility and quick inspection, while the Parquet file supports efficient storage and faster loading for later analytical steps. In both outputs, the explicit `Index` field is retained as the record identifier and the temporary pandas row index is omitted.

In [10]:
# CSV – portable
standardised_text_df.to_csv(Config.CLEAN_CSV_PATH, index=False, encoding="latin-1")

# Parquet – efficient
standardised_text_df.to_parquet(
    Config.CLEAN_PARQUET_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

# Summary of saved files
for path in [Config.RAW_DATA_PATH, Config.CLEAN_CSV_PATH, Config.CLEAN_PARQUET_PATH]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{path.name:<25} | {size_mb:>6.2f} MB")

print("\nAll files saved successfully.")

IT_Tickets_Raw.csv        |  24.79 MB
04_Tickets_Clean.csv      |  21.32 MB
04_Tickets_Clean.parquet  |   9.85 MB

All files saved successfully.
